# LendMind

### 1) Importing Libraries

In [ ]:
!pip install numpy==2.2.0
!pip install pandas==2.2.3
!pip install matplotlib==3.9.3

In [23]:
import numpy as np 
import pandas as pd
from matplotlib import pyplot as plt

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

### 2) Read & Understand The Data

In [24]:
all_data = pd.read_csv(r"..\DataSet\archive\loan.csv")

In [25]:
all_data.shape

(2260668, 145)

A) We will take a random sample from the dataset (all_data), for example 5% of the total dataset:

no of rows = 0.05 * 2260668 = 113,033 rows

--> a)

In [26]:
all_data = all_data[all_data['loan_status'].isin(['Fully Paid', 'Charged Off'])]
all_data['loan_status'] = all_data['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1})
all_data.shape

(1303607, 145)

--> b)

In [27]:
df = all_data.sample(frac= 0.05, random_state= 42)
df.shape

(65180, 145)

In [28]:
print(df.iloc[0:5].to_string())

         id  member_id  loan_amnt  funded_amnt  funded_amnt_inv        term  int_rate  installment grade sub_grade               emp_title emp_length home_ownership  annual_inc verification_status   issue_d  loan_status pymnt_plan  url desc             purpose                    title zip_code addr_state    dti  delinq_2yrs earliest_cr_line  inq_last_6mths  mths_since_last_delinq  mths_since_last_record  open_acc  pub_rec  revol_bal  revol_util  total_acc initial_list_status  out_prncp  out_prncp_inv   total_pymnt  total_pymnt_inv  total_rec_prncp  total_rec_int  total_rec_late_fee  recoveries  collection_recovery_fee last_pymnt_d  last_pymnt_amnt next_pymnt_d last_credit_pull_d  collections_12_mths_ex_med  mths_since_last_major_derog  policy_code application_type  annual_inc_joint  dti_joint verification_status_joint  acc_now_delinq  tot_coll_amt  tot_cur_bal  open_acc_6m  open_act_il  open_il_12m  open_il_24m  mths_since_rcnt_il  total_bal_il  il_util  open_rv_12m  open_rv_24m  max_b

B) dealing with Null values

In [29]:
df = df.loc[:, df.isnull().mean()<0.4]
df.shape

(65180, 87)

In [30]:
print(df.iloc[0:5].to_string())

         loan_amnt  funded_amnt  funded_amnt_inv        term  int_rate  installment grade sub_grade               emp_title emp_length home_ownership  annual_inc verification_status   issue_d  loan_status pymnt_plan             purpose                    title zip_code addr_state    dti  delinq_2yrs earliest_cr_line  inq_last_6mths  open_acc  pub_rec  revol_bal  revol_util  total_acc initial_list_status  out_prncp  out_prncp_inv   total_pymnt  total_pymnt_inv  total_rec_prncp  total_rec_int  total_rec_late_fee  recoveries  collection_recovery_fee last_pymnt_d  last_pymnt_amnt last_credit_pull_d  collections_12_mths_ex_med  policy_code application_type  acc_now_delinq  tot_coll_amt  tot_cur_bal  total_rev_hi_lim  acc_open_past_24mths  avg_cur_bal  bc_open_to_buy  bc_util  chargeoff_within_12_mths  delinq_amnt  mo_sin_old_il_acct  mo_sin_old_rev_tl_op  mo_sin_rcnt_rev_tl_op  mo_sin_rcnt_tl  mort_acc  mths_since_recent_bc  mths_since_recent_inq  num_accts_ever_120_pd  num_actv_bc_tl  num_

C) Remove columns where all cells have the same value (Zero Variance Filter)

In [31]:
df = df.loc[:, df.nunique()>1]
df.shape

(65180, 82)

In [32]:
print(df.iloc[0:10].to_string())

         loan_amnt  funded_amnt  funded_amnt_inv        term  int_rate  installment grade sub_grade                                emp_title emp_length home_ownership  annual_inc verification_status   issue_d  loan_status             purpose                                     title zip_code addr_state    dti  delinq_2yrs earliest_cr_line  inq_last_6mths  open_acc  pub_rec  revol_bal  revol_util  total_acc initial_list_status   total_pymnt  total_pymnt_inv  total_rec_prncp  total_rec_int  total_rec_late_fee  recoveries  collection_recovery_fee last_pymnt_d  last_pymnt_amnt last_credit_pull_d  collections_12_mths_ex_med application_type  acc_now_delinq  tot_coll_amt  tot_cur_bal  total_rev_hi_lim  acc_open_past_24mths  avg_cur_bal  bc_open_to_buy  bc_util  chargeoff_within_12_mths  delinq_amnt  mo_sin_old_il_acct  mo_sin_old_rev_tl_op  mo_sin_rcnt_rev_tl_op  mo_sin_rcnt_tl  mort_acc  mths_since_recent_bc  mths_since_recent_inq  num_accts_ever_120_pd  num_actv_bc_tl  num_actv_rev_tl  num

D) High Cardinality Filter

In [33]:
string_columns = df.select_dtypes(include='object').columns
string_df = df[string_columns]
hated_columns = string_df.loc[:, (string_df.nunique() > 50)].columns
df = df.drop(columns=hated_columns)
df.shape

(65180, 75)

E) Multicollinearity Filter

In [34]:
corr_matrix = df.select_dtypes(include=['number']).corr().abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]
df = df.drop(columns=to_drop)
df.shape

(65180, 66)

F) Preventing Data Leakage